**Learn Machine Learning by Projects** : Based on ML Bookcamp 

[The book](https://www.manning.com/books/machine-learning-bookcamp?a_aid=AGMLBookcamp&a_bid=2eb9ca01) , [Original Repo]( https://github.com/alexeygrigorev/mlbookcamp-code )

<img src="https://images.manning.com/360/480/resize/book/d/f91ead8-e9eb-412e-8f75-5f2d7f588e67/Grigorev-MLB-HI.png" width="200">


# Business Understanding

*__Churn Prediction__*

```identifying customers who are likely to cancel their contracts soon.```

* If the company can do that, it can handle users before churn
* The target variable that we want to predict is categorical and has only two possible outcomes:
churn or not churn (Binary Classification).
* We also would like to understand why the model thinks our customers
churn, and for that, we need to be able to interpret the model’s predictions.

* We will use data from https://www.kaggle.com/blastchar/telco-customer-churn.

* According to the description, this dataset has the following information:
    * __Services of the customers__: phone; multiple lines; internet; tech support and extra services such as online security, backup, device protection, and TV streaming
    * __Account information__: how long they have been clients, type of contract, type of
payment method
    * __Charges__: how much the client was charged in the past month and in total
    * __Demographic information__: gender, age, and whether they have dependents or a partner
    * __Churn__: yes/no, whether the customer left the company within the past month

# Initial Data Preparation

In [ ]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt

In [ ]:
df = pd.read_csv('../input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head().T

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='O')

In [ ]:
df.TotalCharges.mode()[0]

In [ ]:
df[df['TotalCharges']==' ']

In [ ]:
def fix_total_charges(row):
    if row['TotalCharges'] == ' ':
        return row['MonthlyCharges']
    else:
        return row['TotalCharges']
    
df['TotalCharges'] = df.apply(fix_total_charges, axis=1)

In [ ]:
df[df['TotalCharges']==' ']

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
df.isnull().sum()

In [ ]:
df['TotalCharges'].fillna(df['MonthlyCharges'], inplace=True)

## Customer ID
> It is not an important feature, we will drop it

In [ ]:
df.drop('customerID', axis=1, inplace=True)
df.head().T

## Total Charges
> from description, it has white space in 11 rows, we need to handle it

In [ ]:
df.TotalCharges.describe()

In [ ]:
df[df['TotalCharges']==' ']

In [ ]:
len(df[df['TotalCharges']==' '])

In [ ]:
# Convert the "TotalCharges" column to numeric and enforce white spaces to be 'NaN'
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
len(df[df['TotalCharges']==' '])

In [ ]:
# Just Confirm
df.iloc[[488,753,936,1082,1340,3331,3826,4380,5218,6670,6754],:]

## Senior Citizen
> To be treated as category

In [ ]:
df['SeniorCitizen'] = df['SeniorCitizen'].astype('object')

## Some Cleaning

In [ ]:
df.columns = df.columns.str.lower()

In [ ]:
df.columns

## Churn

In [ ]:
df[df.churn == 'Yes']

### Convert `churn` to Numerical Value

In [ ]:
(df.churn == 'Yes').astype(int)

In [ ]:
df.churn = (df.churn == 'Yes').astype(int)

In [ ]:
df.churn 

In [ ]:
df.churn.value_counts()  #imbalance

In [ ]:
df.churn.value_counts(normalize= True)

In [ ]:
df.churn.mean()

# Data Splitting 

### We split the full data into : ( Training Set, Validation Set, Testing Set)
* `Training Set`: to traion our model
* `Validation Set`: to validate and tune the model(s) 
* `Testing Set`: to evaluate the final model (after tuning and selecting the best one)

In [ ]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_valid = train_test_split(df_full_train, test_size=0.2, random_state=1)
print("Training Data Size: ", df_train.shape)
print("Validation Data Size: ", df_valid.shape)
print("Testing Data Size: ", df_test.shape)

## Missing Values

In [ ]:
df_train.isnull().sum()

#### We have missing values in `totalcharges`, we need to find the best value to impute it

In [ ]:
print(df_train['totalcharges'].describe())
df_train['totalcharges'].hist()

#### As the distribution of `totalcharges` is `skewed`, we can impute the missing values with its `median`

#### Note: we calculated the median of `totalcharges` from the `training set` and it will be used for other sets

In [ ]:
df_train.totalcharges.isnull().sum()

In [ ]:
df_valid.totalcharges.isnull().sum()

In [ ]:
df_test.totalcharges.isnull().sum()

In [ ]:
# total_charges_median = df_train['totalcharges'].median()

# # Imputing 
# df_train['totalcharges'] = df_train['totalcharges'].fillna(total_charges_median)
# df_valid['totalcharges'] = df_valid['totalcharges'].fillna(total_charges_median)
# df_test['totalcharges'] = df_test['totalcharges'].fillna(total_charges_median)

# print(df_train.totalcharges.isnull().sum())
# print(df_valid.totalcharges.isnull().sum())
# print(df_test.totalcharges.isnull().sum())

# EDA

#### Note: EDA usually contains many visualizations, but actually we will make it simpler

In [ ]:
df_train.select_dtypes(include=['object']).columns

In [ ]:
categorical = list(df_train.select_dtypes(include=['object']).columns)
numerical = list(df_train.select_dtypes(include=['number']).columns)

In [ ]:
numerical.remove('churn')

In [ ]:
numerical

In [ ]:
df_train[categorical].nunique()

## Feature Importance
``` It’s often done as a part of exploratory data analysis to figure out which variables will be useful for the model.```
 
```It also gives us additional insights about the dataset and helps answer questions like “What makes customers churn?” and “What are the characteristics of people who churn?”```

### Risk Ratio
> risk = group rate / global rate

* ` a group with a risk close to 1 is not risky at all`
* ` a group with a risk lower than 1:, the clients in this group are less likely to churn than clients in general`
* ` a group with a risk higher than 1: there’s more churn in the group than in the population, `

In [ ]:
global_mean = df_train.churn.mean()
round(global_mean, 2)

In [ ]:
df_train.groupby('gender').mean()['churn']

In [ ]:
churn_gender = df_train.groupby('gender').churn.mean()
churn_gender  # we can compare it with the global churn rate

> the difference between the rates is small, the value is not important when predicting churn because this group of customers is not really different from the rest of the customers. 

In [ ]:
churn_partner = df_train.groupby('partner').churn.mean()
churn_partner

> the difference is not small, something inside that group sets it apart from the rest. A machine learning algorithm should be able to pick this up and use it when making predictions.

In [ ]:
gender_risk = churn_gender / global_mean
gender_risk

In [ ]:
partner_risk = churn_partner / global_mean
partner_risk

### Let's conclude `Risk Ratio` for all categorical features in tables

In [ ]:
from IPython.display import display
for feature in categorical:                                           
    df_group = df_train.groupby(by=feature).churn.agg(['mean']) 
    df_group['diff'] = df_group['mean'] - global_mean
    df_group['risk'] = df_group['mean'] / global_mean
    display(df_group)

### Very Useful Tables, but `Visualization` is better

In [ ]:
for feature in categorical[:3]:                                           
    _=sns.countplot(x= feature, hue = 'churn', data=df)
    plt.show()

In [ ]:
for feature in categorical:                                           
    df_group = df_train.groupby(by=feature).churn.agg(['mean']).reset_index()
    graph=sns.barplot(x= feature, y = 'mean', data=df_group, palette='Greens')
    graph.axhline(global_mean, linewidth=3, color='b')
    plt.text(0, global_mean - 0.03, "global_mean", color='black', weight='semibold')
    plt.show()

> ### Some Insights
* `For gender, there is not much difference between females and males.`

* Senior citizens tend to churn more than nonseniors.

* `People with a partner churn less than people with no partner.`

* People who use phone service are not at risk of churning. People who don’t use phone service are even less likely to churn.

* `Clients with no tech support tend to churn more than those who do.`

* People with monthly contracts cancel the contract a lot more often than others, and people with two-year contacts churn very rarely.

### Mutual Information
* More efficient to compare features importance
* Mutual information is a way to `quantify` the degree of dependency between two categorical variables, but it doesn’t work when one of the features is numerical
* `MI(feature;target) = Entropy(feature) - Entropy(feature|target)`
* Get more details about Information Gain and Mutual Information: [Click here](https://machinelearningmastery.com/information-gain-and-mutual-information/) and [here](https://towardsdatascience.com/select-features-for-machine-learning-model-with-mutual-information-534fe387d5c8)

In [ ]:
from sklearn.metrics import mutual_info_score

def calculate_mi(series):
    return mutual_info_score(series, df_train.churn)

df_mi = df_train[categorical].apply(calculate_mi)
df_mi = df_mi.sort_values(ascending=False).to_frame(name='MI')
display(df_mi.head())
display(df_mi.tail())

* `Higher values of mutual information mean a higher degree of dependence: if the mutual information between a categorical variable and the target is high, this categorical variable will be quite useful for predicting the target.` 

* `On the other hand, if the mutual information is low, the categorical variable and the target are independent, and thus the variable will not be useful for predicting the target.`

### Correlation Coefficient
* The correlation coefficient (Pearson’s correlation coefficient). It is a value from –1 to 1
* Positive correlation means that when one variable goes up, the other variable tends to go up as well `(In the case of a binary target, when the values of the variable are high, we see ones more often than zeros. But when the values of the variable are low, zeros become more frequent than ones.)`
* Zero correlation means no relationship between two variables: they are completely independent.
* Negative correlation occurs when one variable goes up and the other goes down. `(In the binary case, if the values are high, we see more zeros than ones in the target variable. When the values are low, we see more ones.)`

In [ ]:
df[numerical].corrwith(df.churn)

#### Tenure
> The correlation between `tenure` and `churn` is –0.35: it has a negative sign, so the longer customers stay, the less often they tend to churn

In [ ]:
t1 =df[df['tenure'] <= 2].churn.mean()
t1

In [ ]:
t2 = df[(df.tenure >= 3) & (df.tenure <= 12)].churn.mean()
t2 

In [ ]:
t3 = df[df['tenure'] >= 12].churn.mean()
t3

In [ ]:
sns.barplot(x =['1-2', '3-12', '+12'], y =[t1,t2,t3], palette='Greens');
plt.title('Churn Rate by Tenure');
plt.xlabel('Tenure');
plt.ylabel('Churn Rate');

#### Monthly Charges
> `monthlycharges` has a positive coefficient of 0.19, which means that customers who pay more tend to leave more often. 

In [ ]:
mc1 =df[df['monthlycharges'] <= 20].churn.mean()
mc1

In [ ]:
mc2 = df[(df.monthlycharges >= 21) & (df.monthlycharges <= 50)].churn.mean()
mc2

In [ ]:
mc3 = df[df['monthlycharges'] > 50].churn.mean()
mc3

In [ ]:
sns.barplot(x =['0-20', '21-50', '+50'], y =[mc1,mc2,mc3], palette='Greens');
plt.title('Churn Rate by Monthly Charges');
plt.xlabel('Monthly Charges');
plt.ylabel('Churn Rate');

#### Total Charges
> `totalcharges` has a negative correlation, which makes sense: the longer people stay with the company, the more they have paid in total, so it’s less likely that they will leave.

In [ ]:
tc1 = df[df['totalcharges'] <= 1000].churn.mean()
tc1

In [ ]:
tc2 = df[(df.totalcharges > 1000) & (df.totalcharges <= 5000)].churn.mean()
tc2

In [ ]:
tc3 = df[df['totalcharges'] > 5000].churn.mean()
tc3

In [ ]:
sns.barplot(x =['0-1000', '1000-5000', '+5000'], y =[tc1,tc2,tc3], palette='Greens');
plt.title('Churn Rate by Total Charges');
plt.xlabel('Total Charges');
plt.ylabel('Churn Rate');

### Note: We will try to train and evaluate our model with `all features` and with `most important ones` and compare both models

# Preprocessing

## Handling Categorical Variables `one-hot encoding`

In [ ]:
# Applying one hot encoding using Pandas

df_train_enc= pd.get_dummies(df_train, drop_first=True)
df_train_enc.head()

> ## Attention
* ### Test data should be set aside prior to preprocessing.
* ### Any statistics such as mean, min and max used for preprocessing should be derived from the training data. 
* ### Otherwise, there will be a data leakage problem.

In [ ]:
# Applying one hot encoding using Sklearn

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer

ohe = OneHotEncoder(drop='first')

transformer = make_column_transformer((ohe, categorical), remainder='passthrough',
                                      verbose_feature_names_out=False)

train_enc = transformer.fit_transform(df_train)
df_train_enc = pd.DataFrame(train_enc, columns=transformer.get_feature_names_out())
df_train_enc

In [ ]:
X_train = df_train_enc.drop('churn', axis=1)
y_train = df_train_enc['churn']

In [ ]:
valid_enc = transformer.transform(df_valid)
df_valid_enc = pd.DataFrame(valid_enc, columns=transformer.get_feature_names_out())
X_valid = df_valid_enc.drop('churn', axis=1)
y_valid = df_valid_enc['churn']

test_enc = transformer.transform(df_test)
df_test_enc = pd.DataFrame(test_enc, columns=transformer.get_feature_names_out())
X_test = df_test_enc.drop('churn', axis=1)
y_test = df_test_enc['churn']

## Scaling Numerical Variables `Standard Scaler`

In [ ]:
df_train[numerical].describe()

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

scaler.fit_transform(df_train[numerical])

In [ ]:
# Transformers
def preprocess(df_train, df_valid, df_test, num, cat):
    # Define transformers
    ohe = OneHotEncoder(drop='first')
    scaler = StandardScaler()

    transformer = make_column_transformer((scaler, num),
                                           (ohe, cat),
                                          remainder='passthrough',verbose_feature_names_out=False)
    # Fitting & Transformation
    X_train = transformer.fit_transform(df_train[cat+num])
    X_valid = transformer.transform(df_valid[cat+num])
    X_test = transformer.transform(df_test[cat+num])
    columns=transformer.get_feature_names_out()
   
    return X_train , X_valid, X_test, columns

In [ ]:
X_train , X_valid, X_test, columns = preprocess(df_train, df_valid, df_test, numerical, categorical)

In [ ]:
X_train

In [ ]:
y_train = df_train['churn']
y_valid = df_valid['churn']
y_test = df_test['churn']

# Modelling
* We will use logistic regression as a classification model

 ## Logistic Regression in breif

* Logistic regression is a linear model, but unlike linear regression, it’s a classification model
* The output of logistic regression is probability; the probability that the observation is positive, or, in other words, the probability that y = 1. `For our case, it’s the probability that the customer will churn.`
* To be able to treat the output as a probability, we need to make sure that the predictions of the model always stay between zero and one. We use a special mathematical function for this purpose called `sigmoid`, and the full formula for the logistic regression model is:

![](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781617296819/files/OEBPS/Images/03_23-Equation_3-3.png)

* The sigmoid function maps any value to a number between zero and one. It’s defined this way:

![](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781617296819/files/OEBPS/Images/03_24.png)

 ### Linear Regression vs. Logistic Regression

```
def linear_regression(xi):
    result = bias
    for j in range(n):
        result = result + xi[j] * w[j]
    return result
```
******

```
def logistic_regression(xi):
    score = bias
    for j in range(n):
        score = score + xi[j] * w[j]
    prob = sigmoid(score)
    return prob
```  

```
import math
def sigmoid(score):
    return 1 / (1 + math.exp(-score))
```

## Applying Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
model_1 = LogisticRegression(solver='liblinear', random_state=1)
model_1.fit(X_train, y_train)  # ====> Weights

In [ ]:
len(columns)

In [ ]:
len(model_1.coef_[0])

In [ ]:
y_val_pred = model_1.predict_proba(X_valid)
y_val_pred

In [ ]:
y_test_pred = model_1.predict_proba(X_test)
y_test_pred

In [ ]:
y_test_pred[:,1]

> This output (probabilities) is often called `soft predictions`. These tell us the probability of churning as a number between zero and one. It’s up to us to decide how to interpret this number and how to use it.

> To get the binary predictions, `hard predictions`, we take the probabilities and cut them above a certain threshold

In [ ]:
y_test_pred[:,1] > 0.5

In [ ]:
y_test_pred = (y_test_pred[:,1] > 0.5).astype(int)
y_test_pred

In [ ]:
y_pred = model_1.predict(X_test)
y_pred

In [ ]:
y_test_pred == y_test

In [ ]:
(y_test_pred == y_test).mean()

In [ ]:
print('LogisticRegression Training Accuracy: ', round(model_1.score(X_train, y_train), 2))
print('LogisticRegression Validation Accuracy: ', round(model_1.score(X_valid, y_valid), 2))
print('LogisticRegression Testing Accuracy: ', round(model_1.score(X_test, y_test), 2))

# Repeat Training and Evaluation for Most Important Features  

In [ ]:
important_cat = df_mi.head().index.to_list()
important_cat

In [ ]:
X_train, X_valid, X_test, columns = preprocess(df_train, df_valid, df_test, numerical, important_cat)

model_2 = LogisticRegression(solver='liblinear', random_state=1)
model_2.fit(X_train, y_train)
print('LogisticRegression Training Accuracy: ', round(model_2.score(X_train, y_train), 2))
print('LogisticRegression Validation Accuracy: ', round(model_2.score(X_valid, y_valid), 2))
print('LogisticRegression Testing Accuracy: ', round(model_2.score(X_test, y_test), 2))

## Using Numerical Features Only

In [ ]:
numerical

In [ ]:
X_train = df_train[numerical]
X_valid = df_valid[numerical]
X_test  = df_test[numerical]

model_3 = LogisticRegression(solver='liblinear', random_state=1)
model_3.fit(X_train, y_train)
print('LogisticRegression Training Accuracy: ', round(model_3.score(X_train, y_train), 2))
print('LogisticRegression Validation Accuracy: ', round(model_3.score(X_valid, y_valid), 2))
print('LogisticRegression Testing Accuracy: ', round(model_3.score(X_test, y_test), 2))

# Model Saving

In [ ]:
import pickle 

pickle.dump(model_2, open("log_reg.pkl", 'wb'))

## Model Loading

In [ ]:
loaded_model = pickle.load(open('log_reg.pkl', 'rb'))

## Model Usage

In [ ]:
df_test[important_cat + numerical].iloc[10]

In [ ]:
# Transformers
def preprocess_fit(df_train, num, cat):
    # Define transformers
    ohe = OneHotEncoder(drop='first')
    scaler = StandardScaler()

    transformer = make_column_transformer((scaler, num),
                                           (ohe, cat),
                                          remainder='passthrough',verbose_feature_names_out=False)
    # Fitting & Transformation
    transformer.fit(df_train[cat+num])
    return transformer

In [ ]:
transformer = preprocess_fit(df_train, numerical, important_cat)

In [ ]:
df_test.iloc[10][important_cat + numerical]

In [ ]:
pd.DataFrame(df_test.iloc[10][important_cat + numerical]).T

In [ ]:
x =transformer.transform(pd.DataFrame(df_test.iloc[10][important_cat + numerical]).T)
x

In [ ]:
loaded_model.predict(x)

In [ ]:
y_test.iloc[10]

In [ ]:
x = df_test.iloc[10][important_cat + numerical].to_dict()

In [ ]:
x

In [ ]:
x = pd.DataFrame(x, index=[0])
x

In [ ]:
transformer.transform(x)

## Transformer Saving

In [ ]:
pickle.dump(transformer, open("transformer.pkl", 'wb'))

# Inference

In [ ]:
trans = pickle.load(open('transformer.pkl', 'rb'))
model = pickle.load(open('log_reg.pkl', 'rb'))

In [ ]:
cust ={'contract': 'Month-to-month',
 'onlinesecurity': 'No',
 'techsupport': 'No',
 'internetservice': 'Fiber optic',
 'onlinebackup': 'Yes',
 'tenure': 32,
 'monthlycharges': 93.95,
 'totalcharges': 2861.45}

In [ ]:
pd.DataFrame(cust, index=[0])

In [ ]:
cust = trans.transform(pd.DataFrame(cust, index=[0]))

In [ ]:
cust

In [ ]:
model.predict(cust)[0]

In [ ]:
if model.predict(cust)[0] == 0:
    print('Not Churn')
else:
    print('Churn')

In [ ]:
model.predict_proba(cust)[0][1]

# Classification Metrics

In [ ]:
from sklearn.metrics import classification_report

y_pred = model_3.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot= True, fmt='0.0f')

In [ ]:
cm

In [ ]:
(Tn, Fp), (Fn, Tp) = cm

In [ ]:
print('True Negative: ', Tn)
print('False Positive: ', Fp)
print('False Negative: ', Fn)
print('True Positive: ', Tp)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print('Accuracy: ', accuracy_score(y_test, y_pred))
print('Precision: ', precision_score(y_test, y_pred))
print('Recall: ', recall_score(y_test, y_pred))
print('F1-Score: ', f1_score(y_test, y_pred))


In [ ]:
print('Precision: ', precision_score(y_test, y_pred, pos_label=0))


In [ ]:
print('Precision: ', precision_score(y_test, y_pred, average='weighted'))
